# Chapter 20: portfolio theory

Use annual decimal means and same-period covariance to check portfolio variance, GMV, A/B/C/D, tangency and existence conditions. Shared analytic functions and the Chapter 9 optimizer use the same inputs. The code saves fresh constructed curves as two.png and frontier.png under RUN_DIR, then separately displays two frozen constructed reference figures. They are distinct generations, not new market frontiers. Both historical French windows are disabled.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch20_portfolio_theory_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys, tempfile

sys.path.insert(0, str(ROOT / 'code'))
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import ch20_portfolio_theory as workflow
from quant_course.portfolio_theory import (global_minimum_variance_weights, minimum_variance_target_weights,
                                           portfolio_moments, tangency_weights)
from quant_course.optimization_tools import solve_mean_variance
print('Project root:', ROOT)

## 1. Double sum and quadratic form

Expand $\sum_i\sum_j w_iw_j\sigma_{ij}$ and compare it with $w^\top\Sigma w$, using exercise 20-E02's inputs.

In [ ]:
vol = np.array([0.10, 0.20, 0.30])
corr = np.array([[1, 0.5, 0], [0.5, 1, -0.2], [0, -0.2, 1]])
cov = np.outer(vol, vol) * corr
w = np.array([0.5, 0.3, 0.2])
double_sum = sum(w[i] * w[j] * cov[i, j] for i in range(3) for j in range(3))
assert np.isclose(double_sum, w @ cov @ w) and np.isclose(double_sum, 0.01126)
print(round(double_sum, 6), round(np.sqrt(double_sum), 4))

## 2. GMV, A/B/C/D, target return and tangency

Inspect shared analytic results. C=muᵀSigma⁻¹mu is dimensionless under a consistent change of return units; A, B and D scale differently.

In [ ]:
two = workflow.two_asset_example()
assert np.allclose(two['gmv']['weights'], [11/14, 3/14])
k = two['constants']
assert np.isclose(k['B'] / k['A'], two['gmv']['mean']) and np.isclose(1 / k['A'], two['gmv']['variance'])
assert np.allclose(two['target_0.08']['weights'], [0.5, 0.5]) and np.isclose(two['target_0.08']['variance'], 0.025)
assert np.allclose(two['tangency']['weights'], [19/34, 15/34])
assert np.isclose(two['tangency']['sharpe'] ** 2, two['max_sharpe_squared_formula'])
{key: (round(v, 6) if isinstance(v, float) else v) for key, v in k.items()}, round(two['tangency']['sharpe'], 5)

## 3. Inefficient branch and tangency conditions

Exercises 20-E06/E07 distinguish the lower branch from the efficient branch and state the domain for a tangency construction. A stationary formula is insufficient without its conditions.

In [ ]:
mu = np.array([0.06, 0.10]); cov2 = np.array(two['covariance'])
low = minimum_variance_target_weights(mu, cov2, target_return=0.05)
mirror = minimum_variance_target_weights(mu, cov2, target_return=2 * k['B'] / k['A'] - 0.05)
assert np.allclose(low, [1.25, -0.25])
assert np.isclose(low @ cov2 @ low, mirror @ cov2 @ mirror)
assert k['B'] - 0.08 * k['A'] < 0
try:
    tangency_weights(mu, cov2, risk_free_return=0.08)
    raise AssertionError('expected rejection')
except ValueError as error:
    print('rf=0.08 rejected:', error)

## 4. Correlation threshold

For a volatility ratio of 0.6, correlation 0.5 permits long-only minimum volatility just below 15%, whereas correlation 1 gives 15%. The diagrams illustrate specified inputs.

In [ ]:
tmp = RUN_DIR
rows = workflow.plot_two_asset(tmp / 'two.png')['min_long_only_volatility_by_rho']
assert rows['+0.5'] < 0.15 and np.isclose(rows['+1.0'], 0.15)
three = workflow.plot_frontier_cml(tmp / 'frontier.png')
print({r: round(v, 4) for r, v in rows.items()})
print('3-asset GMV vol', round(three['gmv_volatility'], 4), 'tangency Sharpe', round(three['tangency_sharpe'], 4))

## 5. Disabled French two-window comparison

The archived optional code verifies six input hashes and compares fitted moments, tangencies and clipping. It is skipped, so the historical chapter tables are not reproduced by default.

In [ ]:
if RUN_FRENCH:
    from quant_course.french_frozen import load_industries_and_rf
    returns, rf, provenance = load_industries_and_rf()
    assert provenance['is_point_in_time'] is False
    summary, weights = workflow.french_windows(returns, rf)
    for label, row in summary.items():
        print(label, {k: round(v, 4) for k, v in row.items()})
    cov = returns.loc['2010-01':'2019-12'].cov().to_numpy()
    g = global_minimum_variance_weights(cov)
    clipped = np.clip(g, 0, None); clipped /= clipped.sum()
    qp = solve_mean_variance(np.zeros(10), cov / np.max(np.abs(cov))).weights
    vol = lambda x: float(np.sqrt(x @ cov @ x))
    assert vol(g) <= vol(qp) <= vol(clipped)
    print('2010s monthly vol: GMV %.3f%%, QP %.3f%%, clipped %.3f%%' % (100*vol(g), 100*vol(qp), 100*vol(clipped)))
    weights.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 6. Interpretation

Inverse covariance can amplify estimation error in mean-based tangency weights. No out-of-sample or after-cost superiority follows; Project 3 treats rolling evaluation separately.

## Reference figures

The horizontal axis is volatility and the vertical axis is the mean over the same period. A cash/risky-fund allocation line is a CAL for specified moments. Historical French in-sample estimates do not demonstrate out-of-sample advantage.

In [ ]:
from IPython.display import Image, display
for name in ["ch20_two_asset_correlation.png", "ch20_frontier_cml.png"]:
    display(Image(filename=str(ROOT / "fixtures/figures/a4" / name)))